# 启动口播工作台

完成仓库部署后，每次实例开机可打开此 Notebook，点击 **运行 → 运行所有单元格**，等待连接完成，再点击 **进入口播工作台**。

Notebook 内核的工作目录须位于本仓库或其子目录；也可在启动 Jupyter 前设置 `REDUB_PROJECT_ROOT=/absolute/path/to/REDUB-STUDIO`。工作台使用原有 Jupyter 登录和数字端口代理路由，需先安装并启用 Jupyter Server Proxy。AutoDL 默认代理前缀为 `/jupyter/`，其他部署可设置 `REDUB_JUPYTER_BASE_URL`。

启动检查最多等待 30 秒，会复用已运行的工作台。若提示仍在启动，稍后再次运行。实例开关机仍由你控制；关闭本页不会关闭实例或停止计费。

In [ ]:
from html import escape
from pathlib import Path
import os
import runpy
from IPython.display import HTML, clear_output, display


def find_project_root():
    configured = os.environ.get("REDUB_PROJECT_ROOT")
    candidates = [Path(configured).expanduser()] if configured else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "apps/redub_studio/redub_studio/server.py").is_file() and (candidate / "deploy/redub_studio/06_start_private_web.py").is_file():
            return candidate
    raise RuntimeError("找不到项目目录。请从仓库目录运行 Notebook，或设置 REDUB_PROJECT_ROOT 为仓库的绝对路径。")


display(HTML('<p>正在连接口播工作台，请稍候……</p>'))
try:
    project_root = find_project_root()
    os.environ["REDUB_PROJECT_ROOT"] = str(project_root)
    helper = project_root / "deploy/redub_studio/06_start_private_web.py"
    port = runpy.run_path(str(helper))["ensure_web"]()
except Exception as error:
    clear_output(wait=True)
    display(HTML('<p style="color:#b42318">连接未完成：' + escape(str(error)) + '</p>'))
else:
    clear_output(wait=True)
    base_url = "/" + os.environ.get("REDUB_JUPYTER_BASE_URL", "/jupyter/").strip("/")
    route = base_url.rstrip("/") + f"/proxy/{port}/"
    display(HTML('<p>口播工作台已就绪。</p><a href="' + escape(route, quote=True) + '" target="_blank" rel="noopener" style="display:inline-block;padding:12px 24px;border-radius:8px;background:#5b37dc;color:white;text-decoration:none;font-size:18px">进入口播工作台</a>'))
